# ARISE — 03 · Topology Anomaly Detection (Substructure Awareness)

The **core contribution** of the paper (Section IV-B1). Topology anomalies act
*collectively*: a group of unrelated nodes forms an unusually **dense substructure**.

**Algorithm**
1. **Region proposal** — for $k = \bar{d} \dots k_{max}$, take the **k-core** of the graph and split it into connected components → candidate substructures $C_j$.
2. **Anomaly degree** — inside $C_j$ compute the **average node-pair cosine similarity** $d_j$ of node embeddings (Eq. 1-2). *Lower* similarity ⇒ more anomalous, so $t_i = 1/d_j$ (Eq. 8).
3. **Multi-size scoring** — weight by $|C_j|$ and average over rounds (Eq. 9). Nodes never in any substructure get score 0.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # make the `arise` package importable
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


We reuse the trained embeddings from notebook 02 by retraining quickly here (self-contained).

In [ ]:
from arise.data import load_dataset, inject_anomalies, PAPER_INJECTION
from arise.pipeline import train_contrastive, node_embeddings, normalize_features

graph = load_dataset("cora")
graph = inject_anomalies(graph, seed=42, **PAPER_INJECTION["Cora"])
gm = normalize_features(graph)
model, _ = train_contrastive(gm, epochs=60, lr=0.003, weight_decay=1e-5, seed=0, verbose=False)
emb = node_embeddings(gm, model)
print("embeddings:", emb.shape)

## 1. Region proposal: k-core substructures

In [ ]:
from arise.topology import detect_substructures
import numpy as np

avg_deg = int(round(graph.adj.sum() / graph.num_nodes))
print("average degree (k_start):", avg_deg)

subs = detect_substructures(graph.adj, k=avg_deg)
print(f"k={avg_deg}: found {len(subs)} substructures; sizes:", sorted([len(s) for s in subs], reverse=True)[:10])

# How many injected topology anomalies land inside detected substructures?
in_sub = set().union(*[set(s.tolist()) for s in subs]) if subs else set()
topo_idx = set(np.where(graph.topo_mask)[0].tolist())
print(f"topology anomalies captured in substructures: {len(topo_idx & in_sub)}/{len(topo_idx)}")

## 2. Average node-pair similarity ⇒ anomaly estimate (Eq. 1-2, 8)
Injected cliques are made of *dissimilar* nodes, so their substructure has **low** average similarity and a **high** $1/d_j$ estimate.

In [ ]:
from arise.topology import _cosine_avg_similarity

rows = []
for s in subs:
    d_j = _cosine_avg_similarity(emb[s])
    n_anom = int(graph.topo_mask[s].sum())
    rows.append((len(s), round(d_j,3), round(1/d_j,3), n_anom))
rows.sort(key=lambda r: -r[2])
print(f"{'size':>5} {'avg_sim':>8} {'1/d_j':>7} {'#topo_anom':>11}")
for r in rows[:12]:
    print(f"{r[0]:>5} {r[1]:>8} {r[2]:>7} {r[3]:>11}")

## 3. Full multi-round topology score (Eq. 9)

In [ ]:
from arise.topology import topology_anomaly_scores
from arise.metrics import evaluate

score_t, info = topology_anomaly_scores(graph.adj, emb)
print(f"rounds run (k={info['k_start']}..): {info['num_rounds']}")
print("topology module on topology anomalies:", evaluate(graph.topo_mask.astype(int), score_t))

In [ ]:
import numpy as np
plt.figure(figsize=(7,4))
m = graph.topo_mask
nz = score_t > 0
plt.hist(score_t[nz & ~m], bins=30, alpha=.6, label="normal (in substructure)", color="#6ea8ff", density=True)
plt.hist(score_t[m], bins=20, alpha=.8, label="topology anomaly", color="#ff5c5c", density=True)
plt.xlabel("topology anomaly score"); plt.ylabel("density"); plt.legend()
plt.title("Topology score separation"); plt.show()

## Summary
- The k-core region proposal captures the injected dense cliques.
- Within those substructures, injected topology anomalies have **low average embedding similarity** → high $1/d_j$ → high topology score.
- This substructure-aware view detects *collective* topology anomalies that node-vs-neighbour methods miss.

➡️ Next: **04** fuses topology + attribute scores (Eq. 12) and evaluates the full ARISE against baselines.